In [1]:
import spacy
import re
import pandas as pd
from spacy.matcher import Matcher

# Load the pre-trained NLP model
nlp = spacy.load('en_core_web_sm')

# Extracted text from the PDF
text = """
Receipt 1264772/111925
Bill number 92911
501524013
Activity 9000 CREATIVE, ARTS AND ENTERTAINMENT ACTIVITIES
Date of entry into operation
Receipt THE GALLERY LINK TATTOO SHOP
Address COUNTRY: Liberia, COUNTY: Montserrado, DISTRICT: DISTRICT #8, CITY/TOWN: BYE-PASS
TIN
The following transactions are in good standing
LBR- OTHERS REGISTRATION FEES
04/27/2023
TAX DUE
#2,000.00#
Operation performed by Comseh Taylor
Monrovia, July 11, 2024
Amount settled $2,000.00 LRD
Amount in words TWO THOUSAND LRD
Edited by LRA
P.O.Box 10 MONROVIA - EMAIL: info@lra.gov.lr
Website: https://revenue.lra.gov.lr
LRA Visa and Stamp
Date of payment 04/28/2023
Received by REPUBLIC OF LIBERIA
Liberia Revenue Authority
cash payment $2,000.00 LRD
"""

# Apply NER :: 
doc = nlp(text)

# Extract and display entities
entities = [(ent.text, ent.label_) for ent in doc.ents]

# Custom regex patterns to capture specific fields
custom_entities = []

# Capture receipt number
receipt_pattern = re.compile(r'Receipt\s(\d+/\d+)')
receipt_match = receipt_pattern.search(text)
if receipt_match:
    custom_entities.append(("Receipt Number", receipt_match.group(1)))

# Capture bill number
bill_pattern = re.compile(r'Bill number\s(\d+)')
bill_match = bill_pattern.search(text)
if bill_match:
    custom_entities.append(("Bill Number", bill_match.group(1)))

# Capture TIN (Tax Identification Number)
tin_pattern = re.compile(r'TIN\s+(\d+)?')
tin_match = tin_pattern.search(text)
if tin_match:
    custom_entities.append(("TIN", tin_match.group(1) or "Not Provided"))

# Capture dates
date_pattern = re.compile(r'(\d{2}/\d{2}/\d{4})')
dates = date_pattern.findall(text)
for date in dates:
    custom_entities.append(("Date", date))

# Capture amounts (both numeric and words)
amount_pattern = re.compile(r'Amount settled \$(\d{1,3}(?:,\d{3})*(?:\.\d{2})?)\sLRD')
amount_match = amount_pattern.search(text)
if amount_match:
    custom_entities.append(("Amount Settled", amount_match.group(1) + " LRD"))

amount_words_pattern = re.compile(r'Amount in words (.+?) LRD')
amount_words_match = amount_words_pattern.search(text)
if amount_words_match:
    custom_entities.append(("Amount in Words", amount_words_match.group(1)))

# Capture address information
address_pattern = re.compile(r'Address (.+)')
address_match = address_pattern.search(text)
if address_match:
    custom_entities.append(("Address", address_match.group(1)))

# Dependency parsing to capture relationships (e.g., operations, dates, and amounts)
for sent in doc.sents:
    for token in sent:
        if token.dep_ == 'nsubj' and token.head.lemma_ == 'perform':
            operation = f"{token.head.text} by {token.text}"
            custom_entities.append(("Operation", operation))

# Combine spaCy entities with custom entities
all_entities = entities + custom_entities

# Create a DataFrame
df = pd.DataFrame(all_entities, columns=['Label', 'Value'])

# Save the DataFrame to a CSV file
df.to_csv('extracted_data.csv', index=False)

# Print the DataFrame
print(df)

                               Label  \
0                            Receipt   
1                              92911   
2                               ARTS   
3                            Receipt   
4                            Liberia   
5                        Montserrado   
6                                  8   
7                               CITY   
8                                TIN   
9                         #2,000.00#   
10           Comseh Taylor\nMonrovia   
11                     July 11, 2024   
12                          2,000.00   
13                               LRD   
14                               TWO   
15                          THOUSAND   
16                               LRD   
17  LRA\nP.O.Box 10 MONROVIA - EMAIL   
18                          LRA Visa   
19                       Stamp\nDate   
20  04/28/2023\nReceived by REPUBLIC   
21                         LIBERIA\n   
22         Liberia Revenue Authority   
23                          2,000.00   


In [2]:
df

,Label,Value
0,Receipt,ORG
1,92911,CARDINAL
2,ARTS,ORG
3,Receipt,PRODUCT
4,Liberia,GPE
5,Montserrado,GPE
6,8,MONEY
7,CITY,ORG
8,TIN,ORG
9,"#2,000.00#",MONEY
